# Titanic Dataset Assignment Walkthrough

This notebook documents the submission pipeline for the Titanic assignment. It shows the cleaning decisions, the engineered features, the transformation plots, and the final selected features used for modeling.

## Part 1: Data Cleaning Decisions

The assignment asks for missing-value handling, outlier handling, consistency checks, and a short explanation of each decision. The next cells load the generated reports and summarize the exact choices used in the scripts.

In [ ]:
import csv
import json
import os
from collections import Counter
from pathlib import Path
from pprint import pprint

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))

import matplotlib.pyplot as plt

possible_roots = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (
        root
        for root in possible_roots
        if (root / "data").exists() and (root / "scripts").exists()
    ),
    Path.cwd(),
)
DATA_DIR = PROJECT_ROOT / "data"


def load_csv(name: str) -> list[dict[str, str]]:
    with (DATA_DIR / name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


def load_json(name: str) -> dict:
    return json.loads((DATA_DIR / name).read_text(encoding="utf-8"))


def preview_columns(rows: list[dict[str, str]], columns: list[str], limit: int = 5) -> list[dict[str, str]]:
    return [{column: row[column] for column in columns} for row in rows[:limit]]


cleaning_report = load_json("cleaning_report.json")
feature_report = load_json("feature_engineering_report.json")
selection_report = load_json("feature_selection_report.json")

train_cleaned = load_csv("train_cleaned.csv")
train_features = load_csv("train_features.csv")
train_selected = load_csv("train_selected.csv")


In [ ]:
cleaning_summary = {
    "missing_before": cleaning_report["missing_values_before"],
    "missing_after": cleaning_report["missing_values_after"],
    "duplicates_removed": cleaning_report["duplicates_removed"],
    "decisions": cleaning_report["decisions"],
    "outlier_bounds": cleaning_report["outlier_bounds"],
}
pprint(cleaning_summary)


### Cleaning Decisions Explained

- `Age` was imputed with a median based on `CleanTitle + Pclass`, then title-level and global fallbacks were used when needed.
- `Fare` was imputed with the median fare of the matching passenger class.
- `Embarked` was filled with the training-set mode, `S`.
- `Cabin` was kept instead of dropped so that deck information could still be extracted later, and `CabinWasMissing` was added as an indicator.
- `Sex` values were normalized to consistent lowercase categories.
- `Age` and `Fare` were capped using IQR bounds to reduce the influence of extreme outliers.

The cleaned training and test outputs contain no remaining blank values, which satisfies the deliverable for `train_cleaned.csv` and supports the later feature engineering steps.

In [ ]:
preview_columns(
    train_cleaned,
    [
        "PassengerId",
        "Age",
        "Fare",
        "Embarked",
        "Cabin",
        "AgeWasMissing",
        "FareWasMissing",
        "CabinWasMissing",
        "CleanTitle",
    ],
)


## Part 2: Feature Engineering

The feature engineering script creates family-based features, title and deck features, transformed numeric features, interaction terms, one-hot encoded variables, and scaled numeric columns.

In [ ]:
engineered_columns = [
    "PassengerId",
    "FamilySize",
    "IsAlone",
    "Deck",
    "AgeGroup",
    "FarePerPerson",
    "LogFare",
    "LogAge",
    "PclassFareInteraction",
    "AgeClassInteraction",
]
preview_columns(train_features, engineered_columns)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

fare_values = [float(row["Fare"]) for row in train_features]
log_fare_values = [float(row["LogFare"]) for row in train_features]
age_values = [float(row["Age"]) for row in train_features]
log_age_values = [float(row["LogAge"]) for row in train_features]

axes[0].hist(fare_values, bins=30, alpha=0.75, color="#1f77b4", label="Fare")
axes[0].hist(log_fare_values, bins=30, alpha=0.55, color="#ff7f0e", label="LogFare")
axes[0].set_title("Fare vs LogFare")
axes[0].set_xlabel("Value")
axes[0].set_ylabel("Passenger count")
axes[0].legend()

axes[1].hist(age_values, bins=25, alpha=0.75, color="#2ca02c", label="Age")
axes[1].hist(log_age_values, bins=25, alpha=0.55, color="#d62728", label="LogAge")
axes[1].set_title("Age vs LogAge")
axes[1].set_xlabel("Value")
axes[1].set_ylabel("Passenger count")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

title_counts = Counter(row["CleanTitle"] for row in train_features)
deck_counts = Counter(row["Deck"] for row in train_features)

axes[0].bar(title_counts.keys(), title_counts.values(), color="#9467bd")
axes[0].set_title("Title Distribution")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(deck_counts.keys(), deck_counts.values(), color="#8c564b")
axes[1].set_title("Deck Distribution")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


### Why These Transformations Were Kept

- `FamilySize` and `IsAlone` capture family context that the raw `SibSp` and `Parch` columns do not summarize directly.
- `CleanTitle` and `Deck` extract signal hidden inside text columns.
- `LogFare` and `LogAge` reduce skew so the numeric ranges are easier for downstream models to learn from.
- Interaction terms such as `PclassFareInteraction` and `AgeClassInteraction` capture effects that may not appear in the individual features alone.
- One-hot encoding was used for nominal categories so the model can treat each category separately.

## Part 3: Feature Selection

The final selection stage ranks numeric features using target correlation plus information gain, then removes redundant features with a pairwise correlation threshold.

In [ ]:
selection_snapshot = {
    "selection_rules": selection_report["selection_rules"],
    "selected_features": selection_report["selected_features"],
    "top_ranked_features": selection_report["ranking"][:10],
}
pprint(selection_snapshot)


In [ ]:
preview_columns(train_selected, ["PassengerId", "Survived", *selection_report["selected_features"][:6]])


### Feature Selection Summary

- `CabinWasMissing` survived the ranking stage, showing that missingness itself is informative.
- Multiple fare-related features were generated, but only the strongest non-redundant ones were kept after correlation filtering.
- Family context remained useful through `FamilySize`, `Parch`, and `IsAlone`.
- The final selected datasets are ready for model training because they keep the target in train, preserve `PassengerId`, and contain no missing values.

## Reproduce The Pipeline

Run the following commands from the project root:

```powershell
python scripts\\data_cleaning.py
python scripts\\feature_engineering.py
python scripts\\feature_selection.py
```

GitHub repo link: `https://github.com/Duke-Dz/titanic_dataset`